In [ ]:
# ============================================================
# CATBOOST REGRESSION - COMPLETE ANALYSIS
# ============================================================
#
# Dataset:
#   Data1-XML.xlsx
#
# Targets:
#   Last 4 columns
#
# Features:
#   All columns before the last 4, excluding PVK material
#
# Main model:
#   CatBoostRegressor
#
# Outputs:
#   1. Evaluation metrics
#   2. Actual vs Predicted plots omitted by request
#   3. Feature Importance
#   4. SHAP Summary
#   5. LIME
#   6. Feature-Target Heatmap
#   7. Excel
#   8. TXT
#
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import os
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from scipy.stats import pearsonr

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from catboost import (
    CatBoostRegressor,
    EFstrType,
    Pool
)

from lime.lime_tabular import (
    LimeTabularExplainer
)


# ============================================================
# 2. SETTINGS
# ============================================================

INPUT_FILE = "Data1-XML.xlsx"

OUTPUT_DIR = "CatBoost_Final_Output"

FIGURE_DIR = os.path.join(
    OUTPUT_DIR,
    "Figures"
)

EXCEL_FILE = os.path.join(
    OUTPUT_DIR,
    "CatBoost_Results.xlsx"
)

TXT_FILE = os.path.join(
    OUTPUT_DIR,
    "CatBoost_Report.txt"
)

RANDOM_STATE = 42

TEST_SIZE = 0.20

DPI = 800

MAX_SHAP_SAMPLES = 500

LIME_NUM_SAMPLES = 5000

N_LIME_FEATURES = 10


# ============================================================
# 3. CREATE OUTPUT DIRECTORIES
# ============================================================

os.makedirs(
    FIGURE_DIR,
    exist_ok=True
)

# Remove stale SHAP summary images from previous runs; fresh plots will be regenerated.
for _old_plot in os.listdir(FIGURE_DIR):
    if _old_plot.startswith("SHAP_Summary_") and _old_plot.lower().endswith(".png"):
        try:
            os.remove(os.path.join(FIGURE_DIR, _old_plot))
        except OSError:
            pass


# ============================================================
# 4. PLOT STYLE
# ============================================================

plt.rcParams["font.family"] = "Times New Roman"

plt.rcParams["font.size"] = 11

plt.rcParams["axes.titlesize"] = 14

plt.rcParams["axes.labelsize"] = 12

plt.rcParams["xtick.labelsize"] = 10

plt.rcParams["ytick.labelsize"] = 10

plt.rcParams["legend.fontsize"] = 10

sns.set_style("whitegrid")


# ============================================================
# 5. LOAD DATA
# ============================================================

print("\nLoading dataset...")

df = pd.read_excel(
    INPUT_FILE
)

print(
    "Original dataset shape:",
    df.shape
)


# ============================================================
# 6. REMOVE EXACT DUPLICATES
# ============================================================

original_samples = len(df)

df = (
    df
    .drop_duplicates()
    .reset_index(drop=True)
)

removed_duplicates = (
    original_samples - len(df)
)

print(
    "Removed duplicate rows:",
    removed_duplicates
)

print(
    "Final dataset shape:",
    df.shape
)


# ============================================================
# 7. TARGETS
# ============================================================

TARGETS = list(
    df.columns[-4:]
)

print("\nTargets:")

for target in TARGETS:

    print(
        "  ",
        target
    )


# ============================================================
# 8. FEATURES — EXCLUDE PVK MATERIAL COMPLETELY
# ============================================================
# PVK material is excluded from model inputs and all downstream
# feature-based analyses (importance, SHAP, LIME, correlations).
# The last four columns remain the target variables.

all_candidate_features = list(df.columns[:-4])

def _normalise_column_name(name):
    # Normalize unusual whitespace, underscores, hyphens and capitalization.
    text = str(name).replace("\u00a0", " ").replace("\u200b", "")
    text = text.strip().lower().replace("_", " ").replace("-", " ")
    return " ".join(text.split())

# Robustly exclude PVK material regardless of capitalization or spacing.
# Match any feature name containing both tokens, e.g. "PVK material",
# "PVK_Material", or "PVK-material".
PVK_COLUMNS = [
    col for col in all_candidate_features
    if ("pvk" in _normalise_column_name(col)
        and "material" in _normalise_column_name(col))
    or _normalise_column_name(col) == "pvk"
]

FEATURES = [
    col for col in all_candidate_features
    if col not in PVK_COLUMNS
]

# Guardrail: this column must never enter X or any feature-based analysis.
assert not any(
    ("pvk" in _normalise_column_name(col)
     and "material" in _normalise_column_name(col))
    or _normalise_column_name(col) == "pvk"
    for col in FEATURES
), f"PVK material was not excluded: {FEATURES}"

if not FEATURES:
    raise ValueError(
        "No model features remain after excluding PVK material. "
        "Please check the input Excel column names."
    )

print("\nExcluded from all analyses (PVK material):", PVK_COLUMNS or "not found")
print("\nFeatures used by the model:")
for feature in FEATURES:
    print("  ", feature)

# No categorical feature is assumed. If the remaining dataset contains
# categorical columns, list them explicitly here; PVK material must not be added.
CATEGORICAL_FEATURES = []


# ============================================================
# 10. CREATE X AND Y
# ============================================================

X = df[
    FEATURES
].copy()

Y = df[
    TARGETS
].copy()


# ============================================================
# 11. CONVERT CATEGORICAL FEATURE TO STRING
# ============================================================

for col in CATEGORICAL_FEATURES:

    X[col] = (
        X[col]
        .astype(str)
    )


# ============================================================
# 12. LOG10 TRANSFORMATION OF DEFECT FEATURES
# ============================================================

DEFECT_FEATURES = [

    col

    for col in FEATURES

    if "defect" in col.lower()

]

print(
    "\nDefect features transformed using log10:"
)

for col in DEFECT_FEATURES:

    print(
        "  ",
        col
    )

    X[col] = np.log10(
        pd.to_numeric(
            X[col]
        )
    )


# ============================================================
# 13. CHECK MISSING VALUES
# ============================================================

missing_X = X.isnull().sum().sum()

missing_Y = Y.isnull().sum().sum()

if missing_X > 0:

    raise ValueError(
        f"X contains {missing_X} missing values."
    )

if missing_Y > 0:

    raise ValueError(
        f"Y contains {missing_Y} missing values."
    )


print(
    "\nNo missing values detected."
)


# ============================================================
# 14. SHUFFLE DATA AND TRAIN / TEST SPLIT
# ============================================================
# Shuffle complete rows while preserving X/Y alignment, then split randomly.
# No 5-fold or 10-fold cross-validation is performed.

_shuffle_rng = np.random.RandomState(RANDOM_STATE)
_shuffle_order = _shuffle_rng.permutation(len(X))
X = X.iloc[_shuffle_order].reset_index(drop=True)
Y = Y.iloc[_shuffle_order].reset_index(drop=True)

X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True
)


print(
    "\nTraining samples:",
    len(X_train)
)

print(
    "Testing samples:",
    len(X_test)
)


# ============================================================
# 15. CATBOOST PARAMETERS
# ============================================================

CATBOOST_PARAMS = {

    # CatBoost training settings.
    "iterations": 2000,

    "depth": 6,

    "learning_rate": 0.03,

    "loss_function": "RMSE",

    "l2_leaf_reg": 5,

    "random_strength": 0.5,

    "random_seed": RANDOM_STATE,

    "verbose": False,

    "thread_count": -1

}


# ============================================================
# 16. METRIC FUNCTION
# ============================================================

def calculate_metrics(
    y_true,
    y_pred
):

    y_true = np.asarray(
        y_true
    )

    y_pred = np.asarray(
        y_pred
    )


    # --------------------------------------------------------
    # MAE
    # --------------------------------------------------------

    mae = mean_absolute_error(
        y_true,
        y_pred
    )


    # --------------------------------------------------------
    # RMSE
    # --------------------------------------------------------

    rmse = np.sqrt(

        mean_squared_error(
            y_true,
            y_pred
        )

    )


    # --------------------------------------------------------
    # R2
    # --------------------------------------------------------

    r2 = r2_score(
        y_true,
        y_pred
    )


    # --------------------------------------------------------
    # MAPE
    # --------------------------------------------------------

    non_zero = (
        np.abs(y_true) > 1e-12
    )

    if np.sum(non_zero) > 0:

        mape = (

            np.mean(

                np.abs(

                    (
                        y_true[non_zero]
                        -
                        y_pred[non_zero]
                    )
                    /
                    y_true[non_zero]

                )

            )
            * 100

        )

    else:

        mape = np.nan


    # --------------------------------------------------------
    # Pearson correlation
    # --------------------------------------------------------

    if len(y_true) >= 2 and np.std(y_true) > 0 and np.std(y_pred) > 0:
        r, p = pearsonr(y_true, y_pred)
    else:
        r, p = np.nan, np.nan


    return {

        "MAE":
            mae,

        "RMSE":
            rmse,

        "R2":
            r2,

        "MAPE (%)":
            mape,

        "Pearson r":
            r,

        "Pearson p-value":
            p

    }


# ============================================================
# 17. STORAGE
# ============================================================

models = {}

train_predictions = {}

test_predictions = {}

evaluation_results = []

feature_importance_results = []

shap_results = []

lime_results = []


# ============================================================
# 19. TRAIN MODELS
# ============================================================

for target in TARGETS:

    print("\n")
    print("=" * 70)
    print(
        f"Training CatBoost model for: {target}"
    )
    print("=" * 70)


    # ========================================================
    # MODEL
    # ========================================================

    model = CatBoostRegressor(

        **CATBOOST_PARAMS

    )


    # Use a shuffled inner validation split to estimate a suitable number
    # of boosting iterations. The external test set is never used for tuning.
    X_fit, X_valid, y_fit, y_valid = train_test_split(
        X_train,
        Y_train[target],
        test_size=0.20,
        random_state=RANDOM_STATE,
        shuffle=True
    )

    tuning_model = CatBoostRegressor(**CATBOOST_PARAMS)
    tuning_model.fit(
        X_fit,
        y_fit,
        cat_features=CATEGORICAL_FEATURES,
        eval_set=(X_valid, y_valid),
        early_stopping_rounds=100,
        verbose=False
    )

    best_iteration = tuning_model.get_best_iteration()
    if best_iteration is None or best_iteration < 1:
        best_iteration = CATBOOST_PARAMS["iterations"]
    else:
        best_iteration += 1

    print(f"Selected iterations for {target}: {best_iteration}")

    # Refit the final model on ALL training rows using the selected iteration
    # count, so the validation subset is not wasted in the final model.
    final_params = dict(CATBOOST_PARAMS)
    final_params["iterations"] = best_iteration
    model = CatBoostRegressor(**final_params)
    model.fit(
        X_train,
        Y_train[target],
        cat_features=CATEGORICAL_FEATURES,
        verbose=False
    )


    models[target] = model


    # ========================================================
    # TRAIN PREDICTION
    # ========================================================

    train_pred = model.predict(
        X_train
    )

    train_predictions[target] = (
        train_pred
    )


    # ========================================================
    # TEST PREDICTION
    # ========================================================

    test_pred = model.predict(
        X_test
    )

    test_predictions[target] = (
        test_pred
    )


    # ========================================================
    # TEST METRICS
    # ========================================================

    metrics = calculate_metrics(

        Y_test[target].values,

        test_pred

    )


    evaluation_results.append({

        "Target":
            target,

        **metrics

    })


    print(
        "\nTest-set performance:"
    )

    for metric_name, value in metrics.items():

        print(
            f"{metric_name}: {value:.6f}"
        )


# ============================================================
# 20. EVALUATION DATAFRAME
# ============================================================

evaluation_df = pd.DataFrame(
    evaluation_results
)


# ============================================================
# 23. ACTUAL VS PREDICTED PLOTS REMOVED BY REQUEST
# Prediction calculations remain available for tables and metrics.


# 24. PREDICTION TABLE
# ============================================================

predictions_df = (

    Y_test

    .reset_index()

    .rename(

        columns={
            "index":
                "Original_Index"
        }

    )

)


for target in TARGETS:

    predictions_df[
        f"{target}_Predicted"
    ] = test_predictions[target]


# ============================================================
# 25. FEATURE IMPORTANCE
# ============================================================

for target in TARGETS:

    model = models[target]


    importance = pd.DataFrame({

        "Target":
            target,

        "Feature":
            FEATURES,

        "Importance":
            model.get_feature_importance()

    })


    importance = importance.sort_values(

        "Importance",

        ascending=False

    )


    feature_importance_results.append(
        importance
    )


    # --------------------------------------------------------
    # Plot
    # --------------------------------------------------------

    plot_data = (

        importance

        .sort_values(
            "Importance",
            ascending=True
        )

    )


    plt.figure(
        figsize=(8, 6)
    )


    sns.barplot(

        data=plot_data,

        x="Importance",

        y="Feature",

        hue="Feature",

        palette="viridis",

        legend=False

    )


    plt.xlabel(
        "CatBoost Feature Importance"
    )

    plt.ylabel(
        "Feature"
    )


    plt.title(

        f"Feature Importance — {target}",

        fontweight="bold"

    )


    plt.tight_layout()


    plt.savefig(

        os.path.join(

            FIGURE_DIR,

            f"Feature_Importance_{target}.png"

        ),

        dpi=DPI,

        bbox_inches="tight"

    )


    plt.close()


feature_importance_df = pd.concat(

    feature_importance_results,

    ignore_index=True

)


# ============================================================
# 26. SHAP
# ============================================================
#
# ONLY ONE STANDARD SHAP PLOT:
# SHAP SUMMARY PLOT
#
# No dependence plots.
#
# ============================================================

print(
    "\n"
    + "=" * 70
)

print(
    "SHAP ANALYSIS"
)

print(
    "=" * 70
)


for target in TARGETS:

    print(
        f"SHAP → {target}"
    )


    model = models[target]


    # --------------------------------------------------------
    # SHAP sample
    # --------------------------------------------------------

    if len(X_test) > MAX_SHAP_SAMPLES:

        X_shap = X_test.sample(

            MAX_SHAP_SAMPLES,

            random_state=RANDOM_STATE

        )

    else:

        X_shap = X_test.copy()


    # --------------------------------------------------------
    # CatBoost native SHAP
    # --------------------------------------------------------

    shap_values = (

        model.get_feature_importance(

            type=EFstrType.ShapValues,

            data=Pool(
                data=X_shap,
                cat_features=CATEGORICAL_FEATURES
            )

        )

    )


    # Last column = expected value
    shap_values_only = (

        shap_values[:, :-1]

    )


    # --------------------------------------------------------
    # SHAP importance
    # --------------------------------------------------------

    mean_abs_shap = (

        np.abs(
            shap_values_only
        )

        .mean(
            axis=0
        )

    )


    shap_importance = pd.DataFrame({

        "Target":
            target,

        "Feature":
            FEATURES,

        "Mean_Abs_SHAP":
            mean_abs_shap

    })


    shap_importance = (

        shap_importance

        .sort_values(

            "Mean_Abs_SHAP",

            ascending=False

        )

    )


    shap_results.append(
        shap_importance
    )


    # --------------------------------------------------------
    # SHAP Summary Plot
    # --------------------------------------------------------

    plt.figure(
        figsize=(8, 6)
    )


    shap.summary_plot(

        shap_values_only,

        X_shap,

        show=False,

        max_display=len(FEATURES)

    )


    plt.title(

        f"SHAP Summary — {target}",

        fontweight="bold"

    )


    plt.tight_layout()


    plt.savefig(

        os.path.join(

            FIGURE_DIR,

            f"SHAP_Summary_{target}.png"

        ),

        dpi=DPI,

        bbox_inches="tight"

    )


    plt.close()


shap_df = pd.concat(

    shap_results,

    ignore_index=True

)


# ============================================================
# 27. LIME
# ============================================================
#
# ONE LIME FIGURE PER TARGET
#
# A representative test sample is selected.
#
# ============================================================

print(
    "\n"
    + "=" * 70
)

print(
    "LIME ANALYSIS"
)

print(
    "=" * 70
)


for target in TARGETS:

    print(
        f"LIME → {target}"
    )


    model = models[target]


    # --------------------------------------------------------
    # Create numeric version for LIME
    # --------------------------------------------------------

    X_train_lime = X_train.copy()

    X_test_lime = X_test.copy()

    # LIME needs numeric input. Encode categorical labels consistently,
    # then decode the generated values back to CatBoost's string categories.
    lime_category_maps = {}
    for col in CATEGORICAL_FEATURES:
        categories = sorted(X_train[col].astype(str).unique().tolist())
        category_to_code = {value: code for code, value in enumerate(categories)}
        code_to_category = {code: value for value, code in category_to_code.items()}
        lime_category_maps[col] = (category_to_code, code_to_category)
        X_train_lime[col] = X_train[col].astype(str).map(category_to_code).astype(float)
        X_test_lime[col] = X_test[col].astype(str).map(category_to_code).astype(float)
        # Test-only unseen categories are not expected after a random split,
        # but fail clearly rather than silently passing NaNs to LIME.
        if X_train_lime[col].isna().any() or X_test_lime[col].isna().any():
            raise ValueError(f"Could not encode categorical feature '{col}' for LIME.")


    # --------------------------------------------------------
    # LIME explainer
    # --------------------------------------------------------

    lime_explainer = LimeTabularExplainer(

        training_data=
            X_train_lime.values,

        feature_names=
            FEATURES,

        mode=
            "regression",

        discretize_continuous=
            True,

        random_state=
            RANDOM_STATE

    )


    # --------------------------------------------------------
    # Select representative test sample
    #
    # Closest sample to median PCE.
    # --------------------------------------------------------

    median_pce = (
        Y_test["PCE"].median()
    )


    representative_position = (

        np.abs(

            Y_test["PCE"].values

            -

            median_pce

        ).argmin()

    )


    sample = (

        X_test_lime

        .iloc[
            representative_position
        ]

    )


    original_index = (

        X_test

        .index[
            representative_position
        ]

    )


    # --------------------------------------------------------
    # Prediction wrapper for LIME
    # --------------------------------------------------------

    def lime_predict(
        data
    ):

        data = pd.DataFrame(

            data,

            columns=FEATURES

        )


        for col in CATEGORICAL_FEATURES:
            category_to_code, code_to_category = lime_category_maps[col]
            max_code = max(code_to_category) if code_to_category else 0
            codes = data[col].round().clip(lower=0, upper=max_code).astype(int)
            data[col] = codes.map(code_to_category).astype(str)


        return model.predict(
            data
        )


    # --------------------------------------------------------
    # Generate explanation
    # --------------------------------------------------------

    explanation = (

        lime_explainer

        .explain_instance(

            sample.values,

            lime_predict,

            num_features=
                N_LIME_FEATURES,

            num_samples=
                LIME_NUM_SAMPLES

        )

    )


    # --------------------------------------------------------
    # Store LIME results
    # --------------------------------------------------------

    for feature_rule, weight in (

        explanation.as_list()

    ):

        lime_results.append({

            "Target":
                target,

            "Sample_Index":
                original_index,

            "Feature_Rule":
                feature_rule,

            "LIME_Weight":
                weight

        })


    # --------------------------------------------------------
    # LIME Figure
    # --------------------------------------------------------

    fig = (
        explanation
        .as_pyplot_figure()
    )


    fig.set_size_inches(
        8,
        6
    )


    plt.title(

        f"LIME Explanation — {target}",

        fontweight="bold"

    )


    plt.tight_layout()


    plt.savefig(

        os.path.join(

            FIGURE_DIR,

            f"LIME_{target}.png"

        ),

        dpi=DPI,

        bbox_inches="tight"

    )


    plt.close(fig)


lime_df = pd.DataFrame(
    lime_results
)


# ============================================================
# 28. FEATURE-TARGET CORRELATION HEATMAP
# ============================================================

# PVK material has already been excluded from FEATURES and X.

X_corr = X.copy()


for col in CATEGORICAL_FEATURES:

    X_corr[col] = (

        X_corr[col]

        .astype(float)

    )


feature_target_corr = (

    pd.concat(

        [
            X_corr,
            Y
        ],

        axis=1

    )

    .corr()

    .loc[
        FEATURES,
        TARGETS
    ]

)


# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

plt.figure(
    figsize=(9, 7)
)


sns.heatmap(

    feature_target_corr,

    annot=True,

    fmt=".2f",

    cmap="RdBu_r",

    center=0,

    linewidths=0.5,

    cbar_kws={

        "label":
            "Pearson Correlation"

    }

)


plt.xlabel(
    "Targets"
)

plt.ylabel(
    "Features"
)


plt.title(

    "Feature–Target Correlation Heatmap",

    fontweight="bold"

)


plt.tight_layout()


plt.savefig(

    os.path.join(

        FIGURE_DIR,

        "Feature_Target_Heatmap.png"

    ),

    dpi=DPI,

    bbox_inches="tight"

)


plt.close()


# ============================================================
# 29. SAVE EXCEL
# ============================================================

print(
    "\nSaving Excel..."
)


with pd.ExcelWriter(

    EXCEL_FILE,

    engine="openpyxl"

) as writer:


    # --------------------------------------------------------
    # Evaluation
    # --------------------------------------------------------

    evaluation_df.to_excel(

        writer,

        sheet_name="Evaluation Metrics",

        index=False

    )


    # --------------------------------------------------------
    # Predictions
    # --------------------------------------------------------

    predictions_df.to_excel(

        writer,

        sheet_name="Test Predictions",

        index=False

    )


    # --------------------------------------------------------
    # Feature Importance
    # --------------------------------------------------------

    feature_importance_df.to_excel(

        writer,

        sheet_name="Feature Importance",

        index=False

    )


    # --------------------------------------------------------
    # SHAP
    # --------------------------------------------------------

    shap_df.to_excel(

        writer,

        sheet_name="SHAP",

        index=False

    )


    # --------------------------------------------------------
    # LIME
    # --------------------------------------------------------

    lime_df.to_excel(

        writer,

        sheet_name="LIME",

        index=False

    )


    # --------------------------------------------------------
    # Correlation
    # --------------------------------------------------------

    feature_target_corr.to_excel(

        writer,

        sheet_name="Feature Target Corr"

    )


# ============================================================
# 30. SAVE TXT REPORT
# ============================================================

print(
    "Saving TXT report..."
)


with open(

    TXT_FILE,

    "w",

    encoding="utf-8"

) as f:


    f.write(
        "=" * 80 + "\n"
    )

    f.write(
        "CATBOOST REGRESSION ANALYSIS REPORT\n"
    )

    f.write(
        "=" * 80 + "\n\n"
    )


    # --------------------------------------------------------
    # Dataset
    # --------------------------------------------------------

    f.write(
        "DATASET INFORMATION\n"
    )

    f.write(
        "-" * 80 + "\n"
    )

    f.write(
        f"Input file: {INPUT_FILE}\n"
    )

    f.write(
        f"Original samples: {original_samples}\n"
    )

    f.write(
        f"Removed duplicates: {removed_duplicates}\n"
    )

    f.write(
        f"Final samples: {len(df)}\n"
    )

    f.write(
        f"Training samples: {len(X_train)}\n"
    )

    f.write(
        f"Testing samples: {len(X_test)}\n"
    )

    f.write(
        f"Number of features: {len(FEATURES)}\n"
    )

    f.write(
        f"Number of targets: {len(TARGETS)}\n"
    )

    f.write(
        "Excluded feature(s): "
        + (", ".join(map(str, PVK_COLUMNS)) if PVK_COLUMNS else "PVK material column not found")
        + "\n"
    )


    # --------------------------------------------------------
    # Evaluation
    # --------------------------------------------------------

    f.write(
        "\n\n"
    )

    f.write(
        "TEST SET EVALUATION\n"
    )

    f.write(
        "-" * 80 + "\n"
    )

    f.write(
        evaluation_df.to_string(
            index=False
        )
    )


    # --------------------------------------------------------
    # Feature Importance
    # --------------------------------------------------------

    f.write(
        "\n\n"
    )

    f.write(
        "FEATURE IMPORTANCE\n"
    )

    f.write(
        "-" * 80 + "\n"
    )

    f.write(
        feature_importance_df.to_string(
            index=False
        )
    )


    # --------------------------------------------------------
    # SHAP
    # --------------------------------------------------------

    f.write(
        "\n\n"
    )

    f.write(
        "SHAP IMPORTANCE\n"
    )

    f.write(
        "-" * 80 + "\n"
    )

    f.write(
        shap_df.to_string(
            index=False
        )
    )


    # --------------------------------------------------------
    # LIME
    # --------------------------------------------------------

    f.write(
        "\n\n"
    )

    f.write(
        "LIME EXPLANATION\n"
    )

    f.write(
        "-" * 80 + "\n"
    )

    f.write(
        lime_df.to_string(
            index=False
        )
    )


# ============================================================
# 31. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 80
)

print(
    "CATBOOST ANALYSIS COMPLETED SUCCESSFULLY"
)

print(
    "=" * 80
)


print(
    "\nEvaluation Metrics:"
)

print(
    evaluation_df.to_string(
        index=False
    )
)


print(
    "\n"
    + "-" * 80
)

print(
    "Excel file:"
)

print(
    EXCEL_FILE
)


print(
    "\nTXT report:"
)

print(
    TXT_FILE
)


print(
    "\nFigures directory:"
)

print(
    FIGURE_DIR
)


print(
    "\nGenerated figures:"
)

print(
    "Actual vs Predicted plots: omitted by request"
)

print(
    "2. Feature Importance × 4"
)

print(
    "3. SHAP Summary × 4"
)

print(
    "4. LIME × 4"
)

print(
    "5. Feature-Target Heatmap × 1"
)


print(
    "\nTotal figures: 13"
)

print(
    "\nDone."
)